In [ ]:
import re
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


SEED = 42
N_SPLITS = 5
TARGET = "triage_acuity"
ID_COL = "patient_id"


# =====================================================
# 1. LOAD
# =====================================================
train = pd.read_csv("/kaggle/input/competitions/triagegeist/train.csv")
test = pd.read_csv("/kaggle/input/competitions/triagegeist/test.csv")
history = pd.read_csv("/kaggle/input/competitions/triagegeist/patient_history.csv")
complaints = pd.read_csv("/kaggle/input/competitions/triagegeist/chief_complaints.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/triagegeist/sample_submission.csv")


# =====================================================
# 2. MERGE ONLY NEW INFORMATION
# =====================================================
# patient_history adds hx_* columns
train = train.merge(history, on=ID_COL, how="left")
test = test.merge(history, on=ID_COL, how="left")

# chief_complaint_system already exists in train/test
# so only bring raw text to avoid duplicated columns
complaints_text = complaints[[ID_COL, "chief_complaint_raw"]].copy()
train = train.merge(complaints_text, on=ID_COL, how="left")
test = test.merge(complaints_text, on=ID_COL, how="left")


# =====================================================
# 3. FEATURE ENGINEERING BASED ON ACTUAL DATA
# =====================================================
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # -------------------------
    # Safe complaint text features
    # -------------------------
    txt = df["chief_complaint_raw"].fillna("").astype(str).str.lower()

    df["cc_len"] = txt.str.len()
    df["cc_word_count"] = txt.str.split().str.len()
    df["cc_has_pain_word"] = txt.str.contains(r"\bpain\b", regex=True).astype(int)
    df["cc_has_chest_word"] = txt.str.contains(r"\bchest\b", regex=True).astype(int)
    df["cc_has_breath_word"] = txt.str.contains(r"breath|dyspnea|shortness", regex=True).astype(int)
    df["cc_has_fever_word"] = txt.str.contains(r"fever", regex=True).astype(int)
    df["cc_has_headache_word"] = txt.str.contains(r"headache", regex=True).astype(int)
    df["cc_has_bleeding_word"] = txt.str.contains(r"bleed|bleeding|hemorrhage", regex=True).astype(int)
    df["cc_has_trauma_word"] = txt.str.contains(r"fall|injury|trauma|fracture", regex=True).astype(int)

    # -------------------------
    # Numeric interaction features
    # -------------------------
    # Dataset already has shock_index, pulse_pressure, news2_score, age_group, arrival_hour
    # so do NOT recreate those.
    if {"heart_rate", "respiratory_rate"}.issubset(df.columns):
        df["hr_rr_ratio"] = df["heart_rate"] / (df["respiratory_rate"] + 1e-3)

    if {"spo2", "respiratory_rate"}.issubset(df.columns):
        df["spo2_rr_interaction"] = df["spo2"] * df["respiratory_rate"]

    if {"age", "news2_score"}.issubset(df.columns):
        df["age_news2_interaction"] = df["age"] * df["news2_score"]

    if {"bmi", "age"}.issubset(df.columns):
        df["bmi_age_interaction"] = df["bmi"] * df["age"]

    # -------------------------
    # Clinical flags
    # -------------------------
    if "temperature_c" in df.columns:
        df["fever_flag"] = (df["temperature_c"] >= 38.0).astype(int)
        df["hypothermia_flag"] = (df["temperature_c"] < 35.0).astype(int)

    if "spo2" in df.columns:
        df["hypoxia_flag"] = (df["spo2"] < 92).astype(int)

    if "heart_rate" in df.columns:
        df["tachycardia_flag"] = (df["heart_rate"] > 100).astype(int)
        df["bradycardia_flag"] = (df["heart_rate"] < 60).astype(int)

    if "respiratory_rate" in df.columns:
        df["tachypnea_flag"] = (df["respiratory_rate"] > 22).astype(int)

    if "systolic_bp" in df.columns:
        df["hypotension_flag"] = (df["systolic_bp"] < 90).astype(int)

    if "gcs_total" in df.columns:
        df["low_gcs_flag"] = (df["gcs_total"] < 15).astype(int)

    if "pain_score" in df.columns:
        df["severe_pain_flag"] = (df["pain_score"] >= 7).astype(int)
        df["pain_missing_flag"] = (df["pain_score"] < 0).astype(int)

    # -------------------------
    # History aggregation
    # -------------------------
    hx_cols = [c for c in df.columns if c.startswith("hx_")]
    if hx_cols:
        df["hx_count"] = df[hx_cols].fillna(0).sum(axis=1)

    return df


train = add_features(train)
test = add_features(test)


# =====================================================
# 4. BUILD TRAIN / TEST MATRICES
# =====================================================
drop_cols = [TARGET, "disposition", "ed_los_hours"]
X = train.drop(columns=drop_cols, errors="ignore").copy()
X_test = test.copy()

# keep exact same feature order
X_test = X_test[X.columns]
y = train[TARGET].copy()


# =====================================================
# 5. DEFINE FEATURE TYPES FROM ACTUAL DATA
# =====================================================
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# treat identifier-like columns as categorical for CatBoost
for c in ["patient_id", "site_id", "triage_nurse_id"]:
    if c in X.columns and c not in cat_cols:
        cat_cols.append(c)

num_cols = [c for c in X.columns if c not in cat_cols]


# =====================================================
# 6. FILL MISSING + FORCE TYPES
# =====================================================
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce")
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

for c in cat_cols:
    X[c] = X[c].fillna("unknown").astype(str)
    X_test[c] = X_test[c].fillna("unknown").astype(str)


# =====================================================
# 7. CV TRAINING
# =====================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof = np.zeros(len(X), dtype=int)
test_proba = np.zeros((len(X_test), y.nunique()))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n===== FOLD {fold} =====")

    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="TotalF1",
        iterations=4000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=8,
        min_data_in_leaf=20,
        random_strength=1.5,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        random_seed=SEED + fold,
        verbose=200
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        use_best_model=True,
        early_stopping_rounds=300
    )

    val_pred = model.predict(X_val).reshape(-1).astype(int)
    oof[val_idx] = val_pred

    test_proba += model.predict_proba(X_test) / N_SPLITS


# =====================================================
# 8. EVALUATION
# =====================================================
cv_score = f1_score(y, oof, average="macro")
print("\nCV Macro F1:", cv_score)


# =====================================================
# 9. SUBMISSION
# =====================================================
final_pred = np.argmax(test_proba, axis=1)

submission = pd.DataFrame({
    ID_COL: sample_sub[ID_COL],
    TARGET: final_pred
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")